# 04 — Scale to the full cohort

**Where you are.** Notebook 03 ended with a working pipeline and proof it
behaves: `process_subject()` ran on 8 age-spread subjects, total cortical
fraction landed 0.551–0.561, and ICV flagged the bad registrations.
Eight subjects is a validation, not a dataset — you can't fit a model
with 48 features on 8 rows, and you can't build age norms from them either.

**The job of this notebook.** Run the *same* pipeline, unchanged, over
every usable subject: **563 rows** (581 scans minus 18 with no age).
The output is one table — `regional_volumes_all.csv` — that the model
notebook (05) will consume. Nothing about the measurement changes here.
What changes is the *engineering*: at roughly a minute or more per
registration, 563 subjects is many hours of compute. A loop that loses
everything when the kernel dies, the laptop sleeps, or subject #412
throws an error is not a plan. So this notebook is built around three
ideas you will reuse in every long batch job you ever run:

1. **Time it first, then plan.** Measure one subject, extrapolate,
   and decide your chunk size from arithmetic, not vibes.
2. **Resume, don't restart.** Save after *every* subject. On re-run,
   skip subjects already in the output file.
3. **Failures are data, not crashes.** One subject's error gets logged
   and the loop moves on; flagged registrations get a second chance
   before they get excluded.

**Plan:** (1) set up + bring in your `process_subject` from notebook 03,
(2) time one subject and do the runtime math, (3) build the work list,
(4) write the resume logic, (5) the chunked batch loop,
(6) QC the results, (7) re-run the flagged subjects once,
(8) read the full table, (9) checkpoint notes.

Like notebook 03, this one must run top-to-bottom on a fresh kernel —
it re-creates the globals your function depends on rather than
assuming notebook 03's kernel is still alive.

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from pathlib import Path
import ants
from nilearn import datasets
import time

subjects_df = pd.read_csv('subjects.csv')
atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')
labels = atlas.labels

template_ants = ants.image_read('data/mni152_template_2mm.nii.gz')
mask_ants = ants.image_read('data/mni152_brain_mask_2mm.nii.gz')
atlas_ants = ants.image_read('data/harvard_oxford_atlas.nii.gz')

def process_subject(path):
    # Stage 1: load the subject
    subject_ants = ants.image_read(path)

    # Stage 2: register it to the template
    reg = ants.registration(fixed=template_ants, moving=subject_ants,type_of_transform='Affine')

    voxel_volume_mm3 = np.prod(subject_ants.spacing)

    # Stage 3: warp the atlas in, count region volumes
    warped_atlas = ants.apply_transforms(fixed=subject_ants, moving=atlas_ants,transformlist=reg['invtransforms'],interpolator='genericLabel')
    warped_arr = warped_atlas.numpy()

    region_volumes = {}
    for region_id in range(1, len(labels)):  # Skip background (0)
        voxel_count = np.sum(warped_arr == region_id)
        volume_ml = (voxel_count * voxel_volume_mm3) / 1000
        region_volumes[labels[region_id]] = volume_ml

    # Stage 4: ICV - warp the brain mask in with the SAME registration
    mask_warped = ants.apply_transforms(fixed=subject_ants, moving=mask_ants,transformlist=reg['invtransforms'],interpolator='genericLabel')
    icv_ml = np.sum(mask_warped.numpy() > 0) * voxel_volume_mm3 / 1000

    # Stage 5: assemble and return
    result_dict = {'icv_ml': icv_ml}
    result_dict.update(region_volumes)
    return result_dict

result = process_subject("data/IXI-T1/IXI002-Guys-0828-T1.nii.gz")
print(result['icv_ml'], result['Frontal Pole'] / result['icv_ml'])

[fetch_atlas_harvard_oxford] Dataset directory found: /Users/jacobruss/nilearn_data/fsl
1136.202462042868 0.06549576669627799


In [ ]:
subjects2_df = subjects_df.dropna(subset=['age'])
subjects2_df = subjects2_df.sort_values(by='age')
# Select subject number 281
subject_281 = subjects2_df.iloc[281]  # 0-based index
file_path = f"data/IXI-T1/{subject_281['file']}"
start = time.perf_counter()
timed_result = process_subject(file_path)
end = time.perf_counter()
print(end - start)


2.5799036249518394


In [17]:
# Timing: one full process_subject() run (IXI342) = 2.58 s
# -> full cohort estimate: 2.58 * 563 ≈ 24 min, one session
print(f"Estimated time for full cohort: {(2.58 * 563) / 60:.1f} minutes")


Estimated time for full cohort: 24.2 minutes


In [ ]:
# Making sure files exist for all subjects in the dataframe
exists_list = []
for _, row in subjects2_df.iterrows():
    path = f"data/IXI-T1/{row['file']}"
    exists_list.append(Path(path).exists())
print(sum(exists_list))


563


In [21]:
# STEP 4 - Resume check: load any existing full-cohort results so the batch skips subjects already processed, verify the saved rows contain no duplicate subjects, and set this session's chunk to everyone remaining.


exists = Path("regional_volumes_all.csv").exists()
if exists:
    results_df = pd.read_csv("regional_volumes_all.csv")
    done = set(results_df['subject'])
else:
    results_df = pd.DataFrame()
    done = set()
chunk = len(subjects2_df) - len(done)
print(f"done: {len(done)}, remaining: {len(subjects2_df) - len(done)}, this session: next {chunk}")

if len(results_df) != len(done):
    raise ValueError("Mismatch between results_df and done set")


done: 0, remaining: 563, this session: next 563


In [ ]:
todo_df = subjects2_df[~subjects2_df['subject'].isin(done)].head(2)
results = results_df.to_dict('records')
failures = []

for _, row in todo_df.iterrows():
    subject_id = row['subject']
    age = row['age']
    file_path = f"data/IXI-T1/{row['file']}"
    try: 
        result_dict = process_subject(file_path)
    except Exception as e:
        print(f"Error processing {subject_id}: {e}")
        failures.append({'subject': subject_id, 'error': str(e)})
        continue
    result_dict['subject'] = subject_id
    result_dict['age'] = age
    print(f"Processed {subject_id}, age {age}, ICV: {result_dict['icv_ml']:.2f} mL", flush=True)
    results.append(result_dict)
    final = pd.DataFrame(results)
    final.to_csv("regional_volumes_all.csv", index=False)

print(f"Failures: {len(failures)}")

if failures:
    failure_df = pd.DataFrame(failures)
    failure_df.to_csv("failures.csv", index=False)

todo length: 2
Processed IXI425, age 19.980835044490075, ICV: 1326.55 mL
Processed IXI064, age 20.071184120465436, ICV: 970.82 mL
Failures: 0


In [ ]:
# STEP 6 - QC the results (runs on a chunk or the full table)
#
# Notebook 03 gave you a provisional rule from 8 subjects. Now you
# have the numbers to look properly. In this step you LOOK - you do
# not delete anything yet.
#
# RECIPE:
# - Read regional_volumes_all.csv fresh from disk (trust the file,
#   not your kernel variables - the file is what notebook 05 gets).
# - Add total_fraction the notebook-03 way: sum the 48 region columns
#   (the label list minus background) into total_ml, divide by icv_ml.
# - Describe icv_ml and total_fraction (pandas .describe()) and
#   print them. Reference points from the pilot: total_fraction for
#   plausible subjects sat 0.551-0.561; pilot ICVs ran 944-1,865,
#   with the two scale families living roughly in the 1,000s-1,300s.
# - Add a qc_flag column using the PROVISIONAL pilot rule:
#   flag subjects with icv_ml < 1000 or icv_ml > 1800. (Those bounds
#   are exactly what flagged IXI191 973, IXI495 944, IXI326 1,865 -
#   and spared the legitimate 1,054-1,327 family members.)
# - Print: how many are flagged, what fraction of the run that is,
#   and the flagged subjects with their age + ICV.
#
# THINK before Step 7: the pilot flagged 3 of 8 (37%). If the full
# run flags a similar share, that is NOT automatically a disaster -
# the two scale families mean some "extreme" ICVs are legitimate
# registrations in the other family. But if a flagged subject's
# total_fraction is ALSO far outside the pack, its region numbers -
# not just its ICV - are untrustworthy. ICV is the smoke alarm;
# total_fraction tells you if there's a fire. Eyeball both columns
# for the flagged rows and write down what you conclude.

In [ ]:
# STEP 7 - Give the flagged subjects ONE second chance
#
# WHY THIS WORKS AT ALL: registration here is stochastic - notebook
# 03 measured the same subject into two different scale families on
# different runs. A failed fit is often not a bad subject, it's a bad
# roll of the dice. Re-running is cheap; excluding a good subject is
# losing data. But ONE re-run, not five: a subject that fails twice
# is telling you something.
#
# RECIPE:
# - For each qc_flagged subject ONLY: call process_subject again,
#   replace their row in the results DataFrame with the new
#   measurements (match on subject), and recompute their flag.
# - Save the updated table back to regional_volumes_all.csv.
# - Print a before/after line per subject: old ICV -> new ICV,
#   flagged -> cleared or still flagged.
# - Subjects STILL flagged after the re-run: leave their rows in the
#   file, flag and all. Notebook 05 will exclude flagged rows from
#   modelling - but the rows, and the reason, stay visible in the
#   data. Deleting them here would make the exclusion unauditable:
#   a reader should be able to ask "where did subject X go?" and
#   find the answer in the table, not in your memory.
#
# Expected: most flagged subjects clear on re-run (they land in a
# scale family with a plausible fraction). A small stubborn residue
# is normal and honest - report its size, don't chase it to zero.

In [ ]:
# STEP 8 - Read the full table (only when done = 563 or close)
#
# Same three plots as notebook 03, Step 7 - the difference is what
# they can say now. With n in the hundreds, clouds become structure:
#
# (a) total cortical FRACTION vs age: the pilot's "gentle downward
#     drift" should now be a visible trend - cortex shrinking as a
#     share of the skull across 20-86. This is the first real
#     scientific signal in the project; look at it properly.
# (b) ICV vs age: must STILL be a flat cloud. More data doesn't
#     change the physiology - skulls don't grow. If (b) trends now,
#     the pipeline is absorbing age into head size and everything
#     downstream is suspect. Check this one FIRST, actually.
# (c) Age histogram of the processed cohort: where are you dense,
#     where are you thin? The thin ends (teens of subjects in their
#     20s and 80s?) are where your future age norms will be shakiest.
#     Know where they are before notebook 05, not after.
#
# RECONCILE, in printed numbers: processed + still-flagged + failed
# must account for all 563 work-list subjects. Write the three counts
# in your checkpoint notes. A subject unaccounted for is a bug; a
# subject accounted for as failed is a result.